In [25]:
import pandas as pd

In [26]:
df = pd.read_csv("../data/nq_first_to_100_predictions_full.csv")
df = df.dropna(how="all").reset_index(drop=True)

In [27]:
df.head()

,Date,Day,Bias,Confidence,Auction Direction,Context,Result,Correct,Notes
0,2026-04-27,Monday,Short,56%,Moderate Down,Exhaustion,Short,True,"HTF structure remains bullish, but the overnig..."
1,2026-04-28,Tuesday,Short,64%,Strong Down,Trend Continuation,Short,True,Large negative ORG and near-ATR overnight sell...
2,2026-04-29,Wednesday,Short,54%,Balanced,Balance,Short,True,Overnight range is compressed and ORG is nearl...
3,2026-04-30,Thursday,Long,63%,Strong Up,Trend Continuation,Short,False,"Strong positive ORG, broad overnight recovery,..."
4,2026-05-01,Friday,Long,61%,Moderate Up,Trend Continuation,Long,True,Higher-timeframe trend remains bullish and pri...


In [28]:
df["Confidence Numeric"] = (
    df["Confidence"]
    .str.rstrip("%")
    .astype(int)
)

In [29]:
confidence_accuracy = (
    df.dropna(subset=["Correct"])
      .groupby("Confidence Numeric")["Correct"]
      .agg(["count", "mean"])
)

confidence_accuracy["Accuracy %"] = confidence_accuracy["mean"] * 100

confidence_accuracy

,count,mean,Accuracy %
Confidence Numeric,,,
53,1,0.0,0.0
54,2,0.5,50.0
55,2,1.0,100.0
56,4,0.75,75.0
57,8,0.625,62.5
58,12,0.583333,58.333333
59,5,0.4,40.0
61,9,0.777778,77.777778
62,7,0.571429,57.142857


In [30]:
valid_df = df.dropna(subset=["Correct"]).copy()

valid_df["Confidence Group"] = pd.cut(
    valid_df["Confidence Numeric"],
    bins=[0, 58, 62, 65, 100],
    labels=["≤58", "59–62", "63–65", "≥66"]
)

In [31]:
confidence_groups = (
    valid_df.groupby("Confidence Group", observed=True)["Correct"]
    .agg(["count", "sum", "mean"])
)

confidence_groups["Accuracy %"] = confidence_groups["mean"] * 100

confidence_groups

,count,sum,mean,Accuracy %
Confidence Group,,,,
≤58,29,18,0.62069,62.068966
59–62,21,13,0.619048,61.904762
63–65,13,6,0.461538,46.153846
≥66,14,8,0.571429,57.142857


In [32]:
valid_df["Confidence Half"] = valid_df["Confidence Numeric"].apply(
    lambda x: "≤62" if x <= 62 else ">62"
)

confidence_halves = (
    valid_df.groupby("Confidence Half")["Correct"]
    .agg(["count", "sum", "mean"])
)

confidence_halves["Accuracy %"] = confidence_halves["mean"] * 100

confidence_halves

,count,sum,mean,Accuracy %
Confidence Half,,,,
>62,27,14,0.518519,51.851852
≤62,50,31,0.62,62.0


In [35]:
overall_accuracy = valid_df["Correct"].mean() * 100

print(f"Overall accuracy: {overall_accuracy:.2f}%")
print(f"Correct: {valid_df['Correct'].sum()} / {len(valid_df)}")

Overall accuracy: 58.44%
Correct: 45 / 77
